In [1]:
!pip install --upgrade pymongo dnspython certifi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 10.3 MB/s eta 0:00:00


In [2]:
from pymongo import MongoClient
from urllib.parse import quote_plus
import certifi
import pandas as pd
import pprint
from datetime import datetime
import os

print("Libraries imported successfully")

Libraries imported successfully


In [3]:


username = "northstar_user"
password = quote_plus("PASSWORD")
cluster_host = "cluster0.hvbklxw.mongodb.net"

connection_string = f"mongodb+srv://{username}:{password}@{cluster_host}/?retryWrites=true&w=majority"

client = MongoClient(
    connection_string,
    tls=True,
    tlsCAFile=certifi.where(),
    serverSelectionTimeoutMS=30000
)

client.admin.command("ping")
print("Connected to MongoDB Atlas successfully")

Connected to MongoDB Atlas successfully


In [4]:
db = client["northstar_db"]

customer_cases = db["customer_cases"]
service_events = db["service_events"]
route_exceptions = db["route_exceptions"]
app_interactions = db["app_interactions"]

print("Database and collections selected successfully")

Database and collections selected successfully


## 1. Load the NorthStar CSV files

Upload the CSV files into the Colab session before running the next cells.

In [5]:
data_path = "/content"
os.listdir(data_path)

['.config', 'sample_data']

In [9]:
orders = pd.read_csv("/content/orders.csv")
deliveries = pd.read_csv("/content/deliveries.csv")
complaints = pd.read_csv("/content/complaints.csv")
customers = pd.read_csv("/content/customers.csv")
drivers = pd.read_csv("/content/drivers.csv")
vehicles = pd.read_csv("/content/vehicles.csv")
hubs = pd.read_csv("/content/hubs.csv")
incidents = pd.read_csv("/content/incidents.csv")
app_events = pd.read_csv("/content/app_events.csv")

print("All CSV files loaded successfully")

All CSV files loaded successfully


In [10]:
orders.head()

,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
0,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
1,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,AIRPORT,Low,109.30,App,0
2,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,AIRPORT,High,33.50,Phone,0
3,O00004,C0520,Parcel,2025-01-11 17:15:00,2,RiverSide,North,Medium,10.04,App,1
4,O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,SOUTH,Low,125.58,Phone,0


In [11]:
deliveries.head()

,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost
0,DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05
1,DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41
2,DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51
3,DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08.103702,Delayed,16.42,0,0,4.18,13.62
4,DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34.131056,OnTime,14.52,1,0,4.18,9.22


In [12]:
complaints.head()

,complaint_id,customer_id,order_id,complaint_type,channel,severity,created_at,status,resolution_days,compensation_amount
0,CP0001,C0464,O00814,AppIssue,App,High,2025-03-30 02:36:00,Open,11,23.99
1,CP0002,C0056,O00628,MissedPickup,Phone,Medium,2024-11-07 10:05:00,Open,4,21.64
2,CP0003,C0469,O00384,Delay,Chatbot,High,2024-01-02 15:47:00,Open,16,26.41
3,CP0004,C0631,O00406,Delay,App,Medium,2025-01-14 13:07:00,AwaitingCustomer,7,23.44
4,CP0005,C0535,O00154,Delay,Email,Medium,2024-08-31 05:56:00,Resolved,1,16.18


## 2. Prepare data for MongoDB

Dates are converted to strings and missing values are converted to `None` to avoid inserting Pandas `NaN` values into MongoDB.

In [13]:
date_columns = {
    "orders": ["order_created_at"],
    "deliveries": ["dispatch_time", "delivery_completed_at"],
    "complaints": ["created_at"],
    "customers": ["signup_date"],
    "vehicles": ["commission_date"],
    "incidents": ["reported_at"],
    "app_events": ["event_timestamp"]
}

dataframes = {
    "orders": orders,
    "deliveries": deliveries,
    "complaints": complaints,
    "customers": customers,
    "drivers": drivers,
    "vehicles": vehicles,
    "hubs": hubs,
    "incidents": incidents,
    "app_events": app_events
}

for df_name, cols in date_columns.items():
    df = dataframes[df_name]
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce").astype(str)


for df_name, df in dataframes.items():
    dataframes[df_name] = df.astype(object).where(pd.notnull(df), None)

orders = dataframes["orders"]
deliveries = dataframes["deliveries"]
complaints = dataframes["complaints"]
customers = dataframes["customers"]
drivers = dataframes["drivers"]
vehicles = dataframes["vehicles"]
hubs = dataframes["hubs"]
incidents = dataframes["incidents"]
app_events = dataframes["app_events"]

print("Date columns converted and missing values prepared for MongoDB")

Date columns converted and missing values prepared for MongoDB


## 3. Clear old collection data

This makes the notebook repeatable. Running it again will not double the document counts.

In [14]:
customer_cases.delete_many({})
service_events.delete_many({})
route_exceptions.delete_many({})
app_interactions.delete_many({})

print("Old documents cleared from MongoDB collections")

Old documents cleared from MongoDB collections


## 4. Create and insert customer case documents

The `customer_cases` collection embeds customer profile, orders, deliveries, complaints, and a case summary.

In [15]:
customer_documents = []

for _, customer in customers.iterrows():
    customer_id = customer["customer_id"]

    customer_orders = orders[orders["customer_id"] == customer_id]
    customer_complaints = complaints[complaints["customer_id"] == customer_id]

    order_list = []

    for _, order in customer_orders.iterrows():
        order_id = order["order_id"]

        delivery_records = deliveries[deliveries["order_id"] == order_id]
        complaint_records = complaints[complaints["order_id"] == order_id]

        order_doc = order.to_dict()
        order_doc["deliveries"] = delivery_records.to_dict("records")
        order_doc["complaints"] = complaint_records.to_dict("records")

        order_list.append(order_doc)

    customer_doc = {
        "customer_id": customer_id,
        "customer_profile": customer.to_dict(),
        "orders": order_list,
        "complaint_history": customer_complaints.to_dict("records"),
        "case_summary": {
            "total_orders": len(customer_orders),
            "total_complaints": len(customer_complaints)
        }
    }

    customer_documents.append(customer_doc)

print("Customer case documents created:", len(customer_documents))

Customer case documents created: 650


In [16]:
if customer_documents:
    result = customer_cases.insert_many(customer_documents)
    print("Inserted customer case documents:", len(result.inserted_ids))
else:
    print("No customer documents to insert")

Inserted customer case documents: 650


In [17]:
sample_customer_case = customer_cases.find_one()
pprint.pprint(sample_customer_case)

{'_id': ObjectId('6a00763a67c01fbb11362db5'),
 'case_summary': {'total_complaints': 2, 'total_orders': 3},
 'complaint_history': [{'channel': 'App',
                        'compensation_amount': 43.9,
                        'complaint_id': 'CP0096',
                        'complaint_type': 'AppIssue',
                        'created_at': '2024-05-12 21:32:00',
                        'customer_id': 'C0001',
                        'order_id': 'O00007',
                        'resolution_days': 22,
                        'severity': 'High',
                        'status': 'Resolved'},
                       {'channel': 'Phone',
                        'compensation_amount': 0.0,
                        'complaint_id': 'CP0146',
                        'complaint_type': 'Delay',
                        'created_at': '2025-09-01 20:17:00',
                        'customer_id': 'C0001',
                        'order_id': 'O00666',
                        'resolution_days': 4,
   

## 5. Create and insert service event documents

The `service_events` collection embeds delivery, driver, vehicle, hub, and incident information.

In [18]:
service_event_documents = []

for _, delivery in deliveries.iterrows():
    driver_info = drivers[drivers["driver_id"] == delivery["driver_id"]]
    vehicle_info = vehicles[vehicles["vehicle_id"] == delivery["vehicle_id"]]
    hub_info = hubs[hubs["hub_id"] == delivery["hub_id"]]
    incident_records = incidents[incidents["delivery_id"] == delivery["delivery_id"]]

    service_doc = {
        "delivery_id": delivery["delivery_id"],
        "order_id": delivery["order_id"],
        "delivery_status": delivery["delivery_status"],
        "dispatch_time": delivery.get("dispatch_time"),
        "delivery_completed_at": delivery.get("delivery_completed_at"),
        "manual_route_override_count": delivery.get("manual_route_override_count"),
        "customer_rating_post_delivery": delivery.get("customer_rating_post_delivery"),
        "driver": driver_info.iloc[0].to_dict() if len(driver_info) > 0 else None,
        "vehicle": vehicle_info.iloc[0].to_dict() if len(vehicle_info) > 0 else None,
        "hub": hub_info.iloc[0].to_dict() if len(hub_info) > 0 else None,
        "incidents": incident_records.to_dict("records")
    }

    service_event_documents.append(service_doc)

print("Service event documents created:", len(service_event_documents))

if service_event_documents:
    result = service_events.insert_many(service_event_documents)
    print("Inserted service event documents:", len(result.inserted_ids))
else:
    print("No service event documents to insert")

Service event documents created: 950
Inserted service event documents: 950


In [19]:
sample_service_event = service_events.find_one()
pprint.pprint(sample_service_event)

{'_id': ObjectId('6a00763f67c01fbb1136303f'),
 'customer_rating_post_delivery': 3.07,
 'delivery_completed_at': '2024-06-19 09:05:59.904311',
 'delivery_id': 'DL00001',
 'delivery_status': 'Failed',
 'dispatch_time': '2024-06-18 10:57:00',
 'driver': {'active_flag': 1,
            'base_zone': 'Airport',
            'driver_id': 'D004',
            'driver_rating': 4.75,
            'employment_type': 'PartTime',
            'shift_preference': 'Morning',
            'training_score': 88.9,
            'years_experience': 13},
 'hub': {'capacity_score': 88,
         'hub_id': 'H05',
         'hub_name': 'Central Core',
         'hub_type': 'Control',
         'zone': 'Central'},
 'incidents': [{'delivery_id': 'DL00001',
                'incident_id': 'I0180',
                'incident_type': 'ProofMissing',
                'reported_at': '2024-06-18 11:38:00',
                'resolution_status': 'Open',
                'resolved_hours': 5.6,
                'severity': 'High'}],
 'man

## 6. Create and insert route exception documents

The `route_exceptions` collection stores deliveries where manual route overrides occurred.

In [20]:
route_exception_documents = []

for _, delivery in deliveries.iterrows():
    override_count = delivery.get("manual_route_override_count")

    if override_count is not None and override_count > 0:
        order_info = orders[orders["order_id"] == delivery["order_id"]]
        hub_info = hubs[hubs["hub_id"] == delivery["hub_id"]]

        route_doc = {
            "delivery_id": delivery["delivery_id"],
            "order_id": delivery["order_id"],
            "delivery_status": delivery["delivery_status"],
            "manual_route_override_count": override_count,
            "dispatch_time": delivery.get("dispatch_time"),
            "delivery_completed_at": delivery.get("delivery_completed_at"),
            "order": order_info.iloc[0].to_dict() if len(order_info) > 0 else None,
            "hub": hub_info.iloc[0].to_dict() if len(hub_info) > 0 else None
        }

        route_exception_documents.append(route_doc)

print("Route exception documents created:", len(route_exception_documents))

if route_exception_documents:
    result = route_exceptions.insert_many(route_exception_documents)
    print("Inserted route exception documents:", len(result.inserted_ids))
else:
    print("No route exception documents to insert")

Route exception documents created: 551
Inserted route exception documents: 551


In [21]:
sample_route_exception = route_exceptions.find_one()
pprint.pprint(sample_route_exception)

{'_id': ObjectId('6a00764367c01fbb113633f5'),
 'delivery_completed_at': '2024-06-19 09:05:59.904311',
 'delivery_id': 'DL00001',
 'delivery_status': 'Failed',
 'dispatch_time': '2024-06-18 10:57:00',
 'hub': {'capacity_score': 88,
         'hub_id': 'H05',
         'hub_name': 'Central Core',
         'hub_type': 'Control',
         'zone': 'Central'},
 'manual_route_override_count': 1,
 'order': {'booking_channel': 'Web',
           'customer_id': 'C0567',
           'dropoff_zone': 'CENTRAL',
           'order_created_at': '2024-06-18 09:48:00',
           'order_id': 'O00938',
           'order_value': 151.14,
           'pickup_zone': 'Central',
           'priority_level': 'Medium',
           'promised_window_hours': 6,
           'service_type': 'Business',
           'special_handling_flag': 0},
 'order_id': 'O00938'}


## 7. Insert app interaction documents

The `app_interactions` collection stores flexible app event records.

In [22]:
app_interaction_documents = app_events.to_dict("records")

if app_interaction_documents:
    result = app_interactions.insert_many(app_interaction_documents)
    print("Inserted app interaction documents:", len(result.inserted_ids))
else:
    print("No app interaction documents to insert")

Inserted app interaction documents: 640


In [23]:
sample_app_interaction = app_interactions.find_one()
pprint.pprint(sample_app_interaction)

{'_id': ObjectId('6a00764567c01fbb1136361c'),
 'api_latency_ms': 301,
 'customer_id': 'C0488',
 'device_type': 'Android',
 'event_id': 'AE00001',
 'event_timestamp': '2024-08-09 03:25:00',
 'event_type': 'eta_refresh',
 'order_id': None,
 'session_id': 'S19847',
 'success_flag': 1,
 'zone_context': 'north'}


In [24]:
print("Customer cases:", customer_cases.count_documents({}))
print("Service events:", service_events.count_documents({}))
print("Route exceptions:", route_exceptions.count_documents({}))
print("App interactions:", app_interactions.count_documents({}))

Customer cases: 650
Service events: 950
Route exceptions: 551
App interactions: 640


## 8. CRUD operations

A demo customer case is inserted, read, updated, verified, and deleted.

In [25]:
new_customer_case = {
    "customer_id": "C_DEMO_001",
    "customer_profile": {
        "customer_id": "C_DEMO_001",
        "customer_segment": "Business",
        "home_zone": "Central",
        "signup_date": "2026-05-10"
    },
    "orders": [
        {
            "order_id": "O_DEMO_001",
            "service_type": "Parcel",
            "pickup_zone": "Central",
            "dropoff_zone": "Airport",
            "delivery_status": "Delayed"
        }
    ],
    "complaint_history": [
        {
            "complaint_id": "CMP_DEMO_001",
            "complaint_type": "Late Delivery",
            "severity": "High",
            "status": "Open"
        }
    ],
    "case_summary": {
        "total_orders": 1,
        "total_complaints": 1
    }
}

customer_cases.delete_many({"customer_id": "C_DEMO_001"})
insert_result = customer_cases.insert_one(new_customer_case)

print("Inserted demo customer case ID:", insert_result.inserted_id)

Inserted demo customer case ID: 6a00764967c01fbb1136389c


In [26]:
demo_customer = customer_cases.find_one({"customer_id": "C_DEMO_001"})
pprint.pprint(demo_customer)

{'_id': ObjectId('6a00764967c01fbb1136389c'),
 'case_summary': {'total_complaints': 1, 'total_orders': 1},
 'complaint_history': [{'complaint_id': 'CMP_DEMO_001',
                        'complaint_type': 'Late Delivery',
                        'severity': 'High',
                        'status': 'Open'}],
 'customer_id': 'C_DEMO_001',
 'customer_profile': {'customer_id': 'C_DEMO_001',
                      'customer_segment': 'Business',
                      'home_zone': 'Central',
                      'signup_date': '2026-05-10'},
 'orders': [{'delivery_status': 'Delayed',
             'dropoff_zone': 'Airport',
             'order_id': 'O_DEMO_001',
             'pickup_zone': 'Central',
             'service_type': 'Parcel'}]}


In [27]:
update_result = customer_cases.update_one(
    {"customer_id": "C_DEMO_001"},
    {
        "$set": {
            "complaint_history.0.status": "Resolved",
            "case_summary.last_updated": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
    }
)

print("Matched documents:", update_result.matched_count)
print("Modified documents:", update_result.modified_count)

Matched documents: 1
Modified documents: 1


In [28]:
updated_customer = customer_cases.find_one(
    {"customer_id": "C_DEMO_001"},
    {"customer_id": 1, "complaint_history": 1, "case_summary": 1}
)

pprint.pprint(updated_customer)

{'_id': ObjectId('6a00764967c01fbb1136389c'),
 'case_summary': {'last_updated': '2026-05-10 12:12:57',
                  'total_complaints': 1,
                  'total_orders': 1},
 'complaint_history': [{'complaint_id': 'CMP_DEMO_001',
                        'complaint_type': 'Late Delivery',
                        'severity': 'High',
                        'status': 'Resolved'}],
 'customer_id': 'C_DEMO_001'}


In [29]:
delete_result = customer_cases.delete_one({"customer_id": "C_DEMO_001"})
print("Deleted documents:", delete_result.deleted_count)

Deleted documents: 1


## 9. Aggregation queries

In [31]:
complaint_summary = customer_cases.aggregate([
    {
        "$group": {
            "_id": "$case_summary.total_complaints",
            "number_of_customers": {"$sum": 1}
        }
    },
    {
        "$sort": {"_id": -1}
    }
])

for item in complaint_summary:
    pprint.pprint(item)

{'_id': 4, 'number_of_customers': 1}
{'_id': 3, 'number_of_customers': 11}
{'_id': 2, 'number_of_customers': 62}
{'_id': 1, 'number_of_customers': 159}
{'_id': 0, 'number_of_customers': 417}


In [30]:
delivery_status_summary = service_events.aggregate([
    {
        "$group": {
            "_id": "$delivery_status",
            "total_deliveries": {"$sum": 1},
            "average_route_overrides": {"$avg": "$manual_route_override_count"},
            "average_customer_rating": {"$avg": "$customer_rating_post_delivery"}
        }
    },
    {
        "$sort": {"total_deliveries": -1}
    }
])

for item in delivery_status_summary:
    pprint.pprint(item)

{'_id': 'OnTime',
 'average_customer_rating': 4.28327302631579,
 'average_route_overrides': 0.9204545454545454,
 'total_deliveries': 616}
{'_id': 'Delayed',
 'average_customer_rating': 3.11497461928934,
 'average_route_overrides': 1.0742574257425743,
 'total_deliveries': 202}
{'_id': 'Failed',
 'average_customer_rating': 3.0493129770992367,
 'average_route_overrides': 1.0378787878787878,
 'total_deliveries': 132}


In [32]:
route_exception_by_hub = route_exceptions.aggregate([
    {
        "$group": {
            "_id": "$hub.hub_name",
            "total_route_exceptions": {"$sum": 1},
            "average_overrides": {"$avg": "$manual_route_override_count"}
        }
    },
    {
        "$sort": {"total_route_exceptions": -1}
    }
])

for item in route_exception_by_hub:
    pprint.pprint(item)

{'_id': 'Midtown Relay',
 'average_overrides': 1.6904761904761905,
 'total_route_exceptions': 84}
{'_id': 'North Exchange',
 'average_overrides': 1.6867469879518073,
 'total_route_exceptions': 83}
{'_id': 'West Gate',
 'average_overrides': 1.5633802816901408,
 'total_route_exceptions': 71}
{'_id': 'Central Core',
 'average_overrides': 1.676923076923077,
 'total_route_exceptions': 65}
{'_id': 'East Dock',
 'average_overrides': 1.6825396825396826,
 'total_route_exceptions': 63}
{'_id': 'Riverside Hub',
 'average_overrides': 1.9206349206349207,
 'total_route_exceptions': 63}
{'_id': 'South Link',
 'average_overrides': 1.564516129032258,
 'total_route_exceptions': 62}
{'_id': 'Airport Hub',
 'average_overrides': 1.5833333333333333,
 'total_route_exceptions': 60}


## 10. Indexing and explain plans

In [33]:
customer_cases.create_index("customer_id")
customer_cases.create_index("case_summary.total_complaints")

service_events.create_index("delivery_id")
service_events.create_index("order_id")
service_events.create_index("delivery_status")

route_exceptions.create_index("delivery_id")
route_exceptions.create_index("manual_route_override_count")

app_interactions.create_index("customer_id")
app_interactions.create_index("event_type")

print("Indexes created successfully")

Indexes created successfully


In [34]:
print("Customer cases indexes:")
pprint.pprint(list(customer_cases.list_indexes()))

print("\nService events indexes:")
pprint.pprint(list(service_events.list_indexes()))

print("\nRoute exceptions indexes:")
pprint.pprint(list(route_exceptions.list_indexes()))

print("\nApp interactions indexes:")
pprint.pprint(list(app_interactions.list_indexes()))

Customer cases indexes:
[SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')]),
 SON([('v', 2), ('key', SON([('customer_id', 1)])), ('name', 'customer_id_1')]),
 SON([('v', 2), ('key', SON([('case_summary.total_complaints', 1)])), ('name', 'case_summary.total_complaints_1')])]

Service events indexes:
[SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')]),
 SON([('v', 2), ('key', SON([('delivery_id', 1)])), ('name', 'delivery_id_1')]),
 SON([('v', 2), ('key', SON([('order_id', 1)])), ('name', 'order_id_1')]),
 SON([('v', 2), ('key', SON([('delivery_status', 1)])), ('name', 'delivery_status_1')])]

Route exceptions indexes:
[SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')]),
 SON([('v', 2), ('key', SON([('delivery_id', 1)])), ('name', 'delivery_id_1')]),
 SON([('v', 2), ('key', SON([('manual_route_override_count', 1)])), ('name', 'manual_route_override_count_1')])]

App interactions indexes:
[SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')]),
 SON

In [35]:
explain_result = customer_cases.find(
    {"customer_id": "C001"}
).explain()

pprint.pprint(explain_result["queryPlanner"]["winningPlan"])

{'inputStage': {'direction': 'forward',
                'indexBounds': {'customer_id': ['["C001", "C001"]']},
                'indexName': 'customer_id_1',
                'indexVersion': 2,
                'isMultiKey': False,
                'isPartial': False,
                'isSparse': False,
                'isUnique': False,
                'keyPattern': {'customer_id': 1},
                'multiKeyPaths': {'customer_id': []},
                'stage': 'IXSCAN'},
 'isCached': False,
 'stage': 'FETCH'}


In [36]:
explain_service_status = service_events.find(
    {"delivery_status": "Failed"}
).explain()

pprint.pprint(explain_service_status["queryPlanner"]["winningPlan"])

{'inputStage': {'direction': 'forward',
                'indexBounds': {'delivery_status': ['["Failed", "Failed"]']},
                'indexName': 'delivery_status_1',
                'indexVersion': 2,
                'isMultiKey': False,
                'isPartial': False,
                'isSparse': False,
                'isUnique': False,
                'keyPattern': {'delivery_status': 1},
                'multiKeyPaths': {'delivery_status': []},
                'stage': 'IXSCAN'},
 'isCached': False,
 'stage': 'FETCH'}


In [37]:
print("Customer cases:", customer_cases.count_documents({}))
print("Service events:", service_events.count_documents({}))
print("Route exceptions:", route_exceptions.count_documents({}))
print("App interactions:", app_interactions.count_documents({}))

Customer cases: 650
Service events: 950
Route exceptions: 551
App interactions: 640
